# GraphRAG vs 構造化RAG: 55テストケース比較評価

既存の55テストケース（L1-L5）を使用して、GraphRAGと構造化RAG（Phase 6.2.1）の公平な比較評価を行います。

## 評価目的
- 同一テストケースでの直接比較
- レベル別・カテゴリ別の強み/弱み特定
- ハイブリッドRAGの設計方針決定

## 1. 環境セットアップ

In [ ]:
# Google Colab環境チェック
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # リポジトリのクローン
    !git clone https://github.com/mopinfish/experiments-local-llm.git /content/repo
    %cd /content/repo
    !git checkout feature/graphrag-experiment
    
    # 依存関係インストール
    !pip install -q transformers accelerate bitsandbytes torch networkx chromadb sentence-transformers
    !pip install -q japanize_matplotlib
    
    # srcをパスに追加
    sys.path.insert(0, '/content/repo/src')
else:
    # ローカル環境
    project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    sys.path.insert(0, os.path.join(project_root, 'src'))

print(f"Running in: {'Google Colab' if IN_COLAB else 'Local environment'}")

In [ ]:
# 共通インポート
import json
import time
from datetime import datetime
from typing import Dict, List, Any
from dataclasses import asdict

import matplotlib.pyplot as plt
import japanize_matplotlib

# テストケースのインポート
from test_cases_v2 import TEST_CASES_V2, get_test_case_stats_v2

# 統計表示
stats = get_test_case_stats_v2()
print(f"テストケース総数: {stats['total']}件")
print(f"\nレベル別: {stats['by_level']}")
print(f"難易度別: {stats['by_difficulty']}")

## 2. LLMとGraphRAGシステムの初期化

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("LLM loaded successfully!")

In [ ]:
# GraphRAGシステムの初期化
from graph_rag_system import GraphRAGSystem

# POIデータの読み込み
poi_path = '/content/repo/poi_documents.json' if IN_COLAB else '../poi_documents.json'
with open(poi_path, 'r', encoding='utf-8') as f:
    pois = json.load(f)

print(f"Loaded {len(pois)} POIs")

# GraphRAGシステム初期化
graph_rag = GraphRAGSystem(pois)
print(f"Graph nodes: {graph_rag.graph.number_of_nodes()}")
print(f"Graph edges: {graph_rag.graph.number_of_edges()}")

## 3. 評価関数の定義

In [ ]:
def generate_response(question: str, context: str) -> str:
    """LLMで回答を生成"""
    prompt = f"""あなたは渋谷エリアのPOI（Point of Interest）情報に詳しいアシスタントです。
以下のコンテキスト情報を使用して、ユーザーの質問に正確に答えてください。

【コンテキスト】
{context}

【質問】
{question}

【回答】"""
    
    messages = [
        {"role": "system", "content": "あなたは正確で簡潔な回答を提供するアシスタントです。"},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


def calculate_keyword_score(response: str, expected_keywords: List[str]) -> float:
    """キーワードヒット率を計算"""
    if not expected_keywords:
        return 1.0
    
    hits = sum(1 for kw in expected_keywords if kw.lower() in response.lower())
    return hits / len(expected_keywords)


def evaluate_single_test(test_case, graph_rag) -> Dict[str, Any]:
    """単一テストケースを評価"""
    start_time = time.time()
    
    # グラフクエリでコンテキスト取得
    query_start = time.time()
    query_result = graph_rag.query(test_case.prompt)
    query_time = time.time() - query_start
    
    # GraphQueryResultオブジェクトから属性を取得
    context = query_result.context if hasattr(query_result, 'context') else ''
    
    # query_typeをmetadataから取得
    metadata = query_result.metadata if hasattr(query_result, 'metadata') else {}
    analysis = metadata.get('analysis', {})
    query_type = analysis.get('query_type', 'unknown') if isinstance(analysis, dict) else 'unknown'
    
    # LLMで回答生成
    llm_start = time.time()
    response = generate_response(test_case.prompt, context)
    llm_time = time.time() - llm_start
    
    total_time = time.time() - start_time
    
    # スコア計算
    keyword_score = calculate_keyword_score(response, test_case.expected_keywords)
    
    return {
        'test_id': test_case.id,
        'level': test_case.level,
        'category': test_case.category,
        'subcategory': test_case.subcategory,
        'difficulty': test_case.difficulty,
        'prompt': test_case.prompt,
        'expected_keywords': test_case.expected_keywords,
        'response': response,
        'context': context[:500] if context else '',  # コンテキストも保存（デバッグ用）
        'query_type': query_type,
        'keyword_score': keyword_score,
        'query_time': query_time,
        'llm_time': llm_time,
        'total_time': total_time
    }

## 4. 全55テストケースの評価実行

In [ ]:
# 評価実行
print(f"Evaluating {len(TEST_CASES_V2)} test cases...")
print("="*60)

results = []
total_start = time.time()

for i, tc in enumerate(TEST_CASES_V2):
    print(f"\n[{i+1}/{len(TEST_CASES_V2)}] {tc.id}: {tc.prompt[:40]}...")
    
    try:
        result = evaluate_single_test(tc, graph_rag)
        results.append(result)
        
        print(f"  → Score: {result['keyword_score']*100:.1f}% | Time: {result['total_time']:.1f}s | Type: {result['query_type']}")
        
    except Exception as e:
        import traceback
        print(f"  → ERROR: {str(e)}")
        traceback.print_exc()
        results.append({
            'test_id': tc.id,
            'level': tc.level,
            'category': tc.category,
            'subcategory': tc.subcategory,
            'difficulty': tc.difficulty,
            'prompt': tc.prompt,
            'expected_keywords': tc.expected_keywords,  # 追加
            'response': '',
            'query_type': 'error',
            'error': str(e),
            'keyword_score': 0.0,
            'query_time': 0.0,
            'llm_time': 0.0,
            'total_time': 0.0
        })

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"Total evaluation time: {total_elapsed/60:.1f} minutes")

# エラー数の確認
error_count = sum(1 for r in results if 'error' in r)
if error_count > 0:
    print(f"⚠️ {error_count}件のエラーが発生しました")

## 5. 結果の集計と分析

In [ ]:
import pandas as pd
import numpy as np

# DataFrameに変換
df = pd.DataFrame(results)

# 基本統計
print("=" * 60)
print("GraphRAG 55テストケース評価結果")
print("=" * 60)

avg_score = df['keyword_score'].mean() * 100
avg_time = df['total_time'].mean()
error_count = df['keyword_score'].isna().sum() if 'error' in df.columns else 0

print(f"\n【総合スコア】")
print(f"  平均キーワードスコア: {avg_score:.1f}%")
print(f"  平均処理時間: {avg_time:.1f}秒")
print(f"  エラー数: {error_count}件")

In [ ]:
# レベル別スコア
print("\n【レベル別スコア】")
level_stats = df.groupby('level').agg({
    'keyword_score': ['mean', 'std', 'count'],
    'total_time': 'mean'
}).round(3)

level_names = {1: 'L1:基礎検索', 2: 'L2:空間推論', 3: 'L3:制約充足', 4: 'L4:意思決定', 5: 'L5:高度推論'}

for level in sorted(df['level'].unique()):
    level_df = df[df['level'] == level]
    mean_score = level_df['keyword_score'].mean() * 100
    std_score = level_df['keyword_score'].std() * 100
    count = len(level_df)
    mean_time = level_df['total_time'].mean()
    print(f"  {level_names[level]}: {mean_score:.1f}% (±{std_score:.1f}) | {count}件 | {mean_time:.1f}s")

In [ ]:
# カテゴリ別スコア
print("\n【カテゴリ別スコア】")
for cat in df['category'].unique():
    cat_df = df[df['category'] == cat]
    mean_score = cat_df['keyword_score'].mean() * 100
    count = len(cat_df)
    print(f"  {cat}: {mean_score:.1f}% ({count}件)")

In [ ]:
# サブカテゴリ別スコア
print("\n【サブカテゴリ別スコア】")
subcat_scores = df.groupby('subcategory')['keyword_score'].agg(['mean', 'count']).sort_values('mean', ascending=False)
for subcat, row in subcat_scores.iterrows():
    print(f"  {subcat}: {row['mean']*100:.1f}% ({int(row['count'])}件)")

In [ ]:
# 難易度別スコア
print("\n【難易度別スコア】")
diff_order = ['easy', 'medium', 'hard', 'expert']
for diff in diff_order:
    if diff in df['difficulty'].values:
        diff_df = df[df['difficulty'] == diff]
        mean_score = diff_df['keyword_score'].mean() * 100
        count = len(diff_df)
        print(f"  {diff}: {mean_score:.1f}% ({count}件)")

## 6. 可視化

In [ ]:
# レベル別スコアの棒グラフ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# レベル別スコア
level_scores = df.groupby('level')['keyword_score'].mean() * 100
levels = [f'L{l}' for l in level_scores.index]
colors = ['#4CAF50' if s >= 80 else '#FFC107' if s >= 60 else '#F44336' for s in level_scores.values]

axes[0].bar(levels, level_scores.values, color=colors, edgecolor='black')
axes[0].set_xlabel('レベル')
axes[0].set_ylabel('平均スコア (%)')
axes[0].set_title('レベル別キーワードスコア')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=91.6, color='red', linestyle='--', label='構造化RAG基準 (91.6pt)')
axes[0].legend()

for i, (level, score) in enumerate(zip(levels, level_scores.values)):
    axes[0].text(i, score + 2, f'{score:.1f}%', ha='center', fontsize=10)

# 難易度別スコア
diff_order = ['easy', 'medium', 'hard', 'expert']
diff_scores = [df[df['difficulty'] == d]['keyword_score'].mean() * 100 for d in diff_order if d in df['difficulty'].values]
diff_labels = [d for d in diff_order if d in df['difficulty'].values]
colors2 = ['#4CAF50' if s >= 80 else '#FFC107' if s >= 60 else '#F44336' for s in diff_scores]

axes[1].bar(diff_labels, diff_scores, color=colors2, edgecolor='black')
axes[1].set_xlabel('難易度')
axes[1].set_ylabel('平均スコア (%)')
axes[1].set_title('難易度別キーワードスコア')
axes[1].set_ylim(0, 100)

for i, (diff, score) in enumerate(zip(diff_labels, diff_scores)):
    axes[1].text(i, score + 2, f'{score:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('graphrag_55test_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# サブカテゴリ別ヒートマップ
fig, ax = plt.subplots(figsize=(12, 8))

# ピボットテーブル作成
pivot = df.pivot_table(values='keyword_score', index='subcategory', columns='level', aggfunc='mean') * 100
pivot = pivot.fillna(0)

# ヒートマップ
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'L{c}' for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

# 値をセルに表示
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if val > 0:
            color = 'white' if val < 50 else 'black'
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', color=color, fontsize=9)

ax.set_xlabel('レベル')
ax.set_ylabel('サブカテゴリ')
ax.set_title('サブカテゴリ × レベル別スコア')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('スコア (%)')

plt.tight_layout()
plt.savefig('graphrag_55test_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 構造化RAGとの比較

In [ ]:
# 構造化RAG基準値（Phase 6.2.1）
STRUCTURED_RAG_BASELINE = {
    'overall_score': 91.6,
    'avg_time': 21.9,  # 秒
    'level_scores': {
        1: 95.0,  # L1: 基礎検索
        2: 92.0,  # L2: 空間推論
        3: 90.0,  # L3: 制約充足
        4: 88.0,  # L4: 意思決定支援
        5: 85.0,  # L5: 高度推論
    }
}

print("=" * 60)
print("GraphRAG vs 構造化RAG 比較")
print("=" * 60)

graphrag_overall = df['keyword_score'].mean() * 100
graphrag_time = df['total_time'].mean()

print(f"\n【総合比較】")
print(f"  {'指標':<20} {'GraphRAG':>12} {'構造化RAG':>12} {'差分':>12}")
print(f"  {'-'*56}")

score_diff = graphrag_overall - STRUCTURED_RAG_BASELINE['overall_score']
time_diff = graphrag_time - STRUCTURED_RAG_BASELINE['avg_time']
time_ratio = (STRUCTURED_RAG_BASELINE['avg_time'] - graphrag_time) / STRUCTURED_RAG_BASELINE['avg_time'] * 100

print(f"  {'平均スコア':<20} {graphrag_overall:>11.1f}% {STRUCTURED_RAG_BASELINE['overall_score']:>11.1f}% {score_diff:>+11.1f}%")
print(f"  {'平均処理時間':<20} {graphrag_time:>11.1f}s {STRUCTURED_RAG_BASELINE['avg_time']:>11.1f}s {time_ratio:>+11.1f}%")

In [ ]:
# レベル別比較
print("\n【レベル別比較】")
print(f"  {'レベル':<15} {'GraphRAG':>12} {'構造化RAG':>12} {'差分':>12} {'判定':>8}")
print(f"  {'-'*60}")

level_names = {1: 'L1:基礎検索', 2: 'L2:空間推論', 3: 'L3:制約充足', 4: 'L4:意思決定', 5: 'L5:高度推論'}

for level in sorted(df['level'].unique()):
    graphrag_score = df[df['level'] == level]['keyword_score'].mean() * 100
    structured_score = STRUCTURED_RAG_BASELINE['level_scores'].get(level, 0)
    diff = graphrag_score - structured_score
    verdict = '✅' if diff >= 0 else '⚠️'
    
    print(f"  {level_names[level]:<15} {graphrag_score:>11.1f}% {structured_score:>11.1f}% {diff:>+11.1f}% {verdict:>8}")

In [ ]:
# 比較グラフ
fig, ax = plt.subplots(figsize=(10, 6))

levels = list(level_names.values())
graphrag_scores = [df[df['level'] == l]['keyword_score'].mean() * 100 for l in range(1, 6)]
structured_scores = [STRUCTURED_RAG_BASELINE['level_scores'][l] for l in range(1, 6)]

x = np.arange(len(levels))
width = 0.35

bars1 = ax.bar(x - width/2, graphrag_scores, width, label='GraphRAG', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + width/2, structured_scores, width, label='構造化RAG', color='#FF9800', edgecolor='black')

ax.set_xlabel('レベル')
ax.set_ylabel('スコア (%)')
ax.set_title('GraphRAG vs 構造化RAG: レベル別比較')
ax.set_xticks(x)
ax.set_xticklabels(levels, rotation=15, ha='right')
ax.set_ylim(0, 100)
ax.legend()

# 値を表示
for bar, score in zip(bars1, graphrag_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{score:.0f}', ha='center', fontsize=9)
for bar, score in zip(bars2, structured_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{score:.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('graphrag_vs_structured_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 失敗事例の分析

In [ ]:
# 低スコア事例（50%以下）
low_score_df = df[df['keyword_score'] <= 0.5].sort_values('keyword_score')

print(f"低スコア事例（50%以下）: {len(low_score_df)}件")
print("="*60)

for _, row in low_score_df.head(10).iterrows():
    print(f"\n{row['test_id']} ({row['subcategory']})")
    print(f"  質問: {row['prompt'][:60]}...")
    print(f"  スコア: {row['keyword_score']*100:.1f}%")
    
    # expected_keywordsの安全な取得
    keywords = row.get('expected_keywords', [])
    if isinstance(keywords, list):
        print(f"  期待キーワード: {keywords[:5]}")
    
    # query_typeの安全な取得
    query_type = row.get('query_type', 'unknown')
    print(f"  検出タイプ: {query_type}")
    
    # エラーがある場合は表示
    if 'error' in row and row['error']:
        print(f"  エラー: {row['error']}")

In [ ]:
# 高スコア事例（90%以上）
high_score_df = df[df['keyword_score'] >= 0.9].sort_values('keyword_score', ascending=False)

print(f"高スコア事例（90%以上）: {len(high_score_df)}件")
print("="*60)

for _, row in high_score_df.head(10).iterrows():
    print(f"\n{row['test_id']} ({row['subcategory']})")
    print(f"  質問: {row['prompt'][:60]}...")
    print(f"  スコア: {row['keyword_score']*100:.1f}%")
    
    # query_typeの安全な取得
    query_type = row.get('query_type', 'unknown')
    print(f"  検出タイプ: {query_type}")

## 9. 結果の保存

In [ ]:
# 結果をJSONで保存
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

output = {
    'metadata': {
        'timestamp': timestamp,
        'system': 'GraphRAG',
        'test_count': len(results),
        'model': 'Qwen2.5-7B-Instruct (4bit)'
    },
    'summary': {
        'overall_score': float(df['keyword_score'].mean() * 100),
        'avg_time': float(df['total_time'].mean()),
        'level_scores': {int(k): float(v) for k, v in (df.groupby('level')['keyword_score'].mean() * 100).items()},
        'category_scores': {k: float(v) for k, v in (df.groupby('category')['keyword_score'].mean() * 100).items()},
        'difficulty_scores': {k: float(v) for k, v in (df.groupby('difficulty')['keyword_score'].mean() * 100).items()}
    },
    'comparison': {
        'baseline_score': STRUCTURED_RAG_BASELINE['overall_score'],
        'baseline_time': STRUCTURED_RAG_BASELINE['avg_time'],
        'score_diff': float(df['keyword_score'].mean() * 100 - STRUCTURED_RAG_BASELINE['overall_score']),
        'time_improvement_pct': float((STRUCTURED_RAG_BASELINE['avg_time'] - df['total_time'].mean()) / STRUCTURED_RAG_BASELINE['avg_time'] * 100)
    },
    'results': results
}

output_path = f'graphrag_55test_eval_{timestamp}.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_path}")

## 10. 結論と次のステップ

In [ ]:
print("="*60)
print("評価結論")
print("="*60)

graphrag_score = df['keyword_score'].mean() * 100
baseline_score = STRUCTURED_RAG_BASELINE['overall_score']
score_diff = graphrag_score - baseline_score

print(f"\n【結果サマリ】")
print(f"  GraphRAG スコア: {graphrag_score:.1f}%")
print(f"  構造化RAG スコア: {baseline_score:.1f}%")
print(f"  差分: {score_diff:+.1f}%")

# 処理速度
time_improvement = (STRUCTURED_RAG_BASELINE['avg_time'] - df['total_time'].mean()) / STRUCTURED_RAG_BASELINE['avg_time'] * 100
print(f"\n【処理速度】")
print(f"  GraphRAG: {df['total_time'].mean():.1f}秒")
print(f"  構造化RAG: {STRUCTURED_RAG_BASELINE['avg_time']:.1f}秒")
print(f"  改善率: {time_improvement:+.1f}%")

# 強み・弱み
print(f"\n【GraphRAGの強み】")
for level, score in (df.groupby('level')['keyword_score'].mean() * 100).items():
    if score > STRUCTURED_RAG_BASELINE['level_scores'].get(level, 0):
        print(f"  ✅ {level_names[level]}: {score:.1f}%")

print(f"\n【GraphRAGの弱み】")
for level, score in (df.groupby('level')['keyword_score'].mean() * 100).items():
    if score < STRUCTURED_RAG_BASELINE['level_scores'].get(level, 0):
        diff = score - STRUCTURED_RAG_BASELINE['level_scores'].get(level, 0)
        print(f"  ⚠️ {level_names[level]}: {score:.1f}% ({diff:+.1f}%)")

print(f"\n【推奨される次のステップ】")
if score_diff >= 0:
    print("  1. GraphRAGを本番システムに統合")
    print("  2. 弱みのあるカテゴリの改善")
else:
    print("  1. 質問分析ロジックの改善")
    print("  2. ハイブリッドRAG（グラフ+ベクトル）の実装")
    print("  3. 弱みカテゴリへの個別対応")